# 07 自定义Skills技能


## 1. 本篇对应的课程与阅读范围

前两篇已经整理了代码存档、远端同步、分支开发和代码审查。本篇继续解决一个问题：同一套检查或处理步骤反复使用时，怎样保存为 Claude Code 能调用的技能，减少每次重新描述流程的成本。

本篇对应 `16-工程化控制-自定义skills技能.mp4`。课程文件见 [VibeCoding实战网盘](https://pan.baidu.com/s/1zQT5QFTIAOSm-e5cnSYy5A?pwd=1234)，提取码 `1234`；主要依据 [Part2 工程化控制和Agent开发](https://www.yuque.com/xxcls/vibecoding/ia0yu2m1v3sl537p) 的 4.1“技能系统概述”和 4.2“创建自定义技能”。

已读取相关讲义正文与原始代码块，保留其中的指令模板和 `my-skill` lint 示例。视频未逐段观看或转录，不推断课堂实际使用的技能名称、目录和执行结果。

讲义中的 `settings.json` 注册方式和插件目录示例与当前官方技能结构存在差异，下面先保留原文意图，再就地说明差异。当前写法依据 [Claude Code Skills 文档](https://code.claude.com/docs/en/skills)，核对日期为 `2026-10-10`。这些修正和与记账项目衔接的内容明确标为补充。

本篇讲的是 **Claude Code 的 Skills**，不修改当前电脑上的 Codex 技能。后续单元测试技能、测试专家 Subagent 和 Hook 配置按课程顺序分别留在 08、09。


## 2. Skills：把重复使用的做法保存下来

讲义将 Skills 作为 Claude Code 的扩展机制，分为内置技能和自定义技能。自定义技能可以通过 `/skill-name` 这样的入口调用。

一个典型过程是：先用普通对话完成一次任务，发现后续还要重复相同的检查顺序、判断条件和输出格式，再把这些内容写进技能文件。

例如每次做 lint 检查，都希望 Claude：运行检查 → 按问题类型整理 → 修复可机械处理的问题 → 对行为相关问题说明影响 → 再次检查。把这套流程保存后，可以用 `/my-skill` 调用，而不必重新粘贴整段要求。

**补充：**技能正文主要是给模型的说明，不是已经训练进模型的新能力，也不是固定执行的程序。Claude 读取说明后仍需使用文件、终端等工具完成任务；实际结果取决于项目条件、工具权限和说明质量。

要得到可靠结果，技能应明确：何时使用、先检查什么、怎样操作、失败时怎么办，以及最后用什么证据判断完成。只有一个名字或“帮我检查项目”这样的泛泛要求，不能代替具体流程。


## 3. 与 CLAUDE.md、普通提示词怎样配合

补充区分三个常用位置，避免把所有说明都塞进同一个文件：

| 位置 | 适合保存的内容 | 记账项目中的例子 |
|---|---|---|
| 当前对话 | 本次临时目标、输入和反馈 | 本次只修复某个筛选问题及其复现步骤 |
| `CLAUDE.md` | 持续适用的项目约定 | 沿用 Tauri + React + TypeScript、IndexedDB + Dexie |
| `SKILL.md` | 可重复调用的任务流程或专项参考 | 按固定顺序检查、分类问题、复验并报告 |

按照当前官方说明，通常先提供技能名称与描述供 Claude 判断相关性，完整正文在调用时进入上下文。设置为仅手动调用的技能，其自动发现行为又有所不同，后文会说明。

技能不适合复制整段历史对话，也不应把每一次运行输出都永久追加进去。保留稳定步骤；本次目标、文件范围和运行结果留在具体任务中。

技能也不必重复抄写所有 `CLAUDE.md` 约定，可以要求先读取相关项目说明。这样项目约定变化时，更容易保持一致。


## 4. 讲义方式一：把简单技能理解为指令模板

讲义先给出 `settings.json` 中的示例：

```json
{
  "skills": {
    "test": "Run the project tests. Use: npm test",
    "deploy-staging": "Deploy to staging. Steps: 1) npm run build 2) npm run deploy:staging 3) Verify at staging.example.com"
  }
}
```

讲义列出的配置位置是项目级 `<project>/.claude/settings.local.json` 和用户级 `~/.claude/settings.json`。

这个例子的教学意图很清楚：把“运行测试”和“构建、部署、验证”等重复指令取一个名字，作为可复用入口。`staging.example.com` 是示例地址，`npm run deploy:staging` 也只是示例脚本，不代表记账项目已经配置了部署。

**当前写法差异：**当前官方 Skills 文档使用技能目录中的 `SKILL.md` 定义技能；没有将上面这种根级 `skills` 对象列为通用注册方式。不能只把这段 JSON 粘进当前配置，就假定 `/test` 和 `/deploy-staging` 一定出现。

本篇保留这段原文用于理解指令模板，实践部分采用当前的技能文件结构。实际项目若还使用了自定义封装，应以该封装明确支持的配置为准。

`settings.json` 与 `settings.local.json` 仍是设置文件，但**设置文件存在**不等于**讲义中的任意字段都会生效**。也不要为创建技能而覆盖已有设置。


## 5. 讲义方式二：技能文件的头部与执行步骤

讲义的第二种方式给出以下目录：

```text
~/.claude/plugins/
└── my-skill/
    └── skill.md          # 技能定义
```

随后给出文件正文：

````markdown
---
name: my-skill
description: 运行自定义的 lint 检查并自动修复
---

# my-skill

## 触发条件
当用户输入 `/my-skill` 或提到 "运行 lint 检查" 时触发。

## 执行步骤
1. 运行 `npm run lint` 获取所有 lint 错误
2. 按错误类型分组
3. 对于可自动修复的错误（如 prettier 格式），自动应用修复
4. 对于需要手动判断的错误，逐一向用户确认
5. 修复后再次运行 lint 确认无错误
````

这个例子包含两层信息：头部说明名称和用途；正文说明触发场景与操作顺序。它强调的不只是检查，还包括**分类、处理和复查**。

**当前结构差异：**普通个人技能放在 `~/.claude/skills/<技能名>/SKILL.md`，项目技能放在 `.claude/skills/<技能名>/SKILL.md`。正式插件里的技能通常在 `<插件>/skills/<技能名>/SKILL.md`，还需要插件自己的加载机制；并不是在 `~/.claude/plugins/` 下随意放一个 `skill.md` 就完成注册。

因此下面沿用 `my-skill` 的教学流程，修正存放位置，并补齐项目脚本和修复范围的前提。文件名统一使用 `SKILL.md`，避免依赖 Windows 对大小写的宽松处理。


## 6. 补充：选项目技能还是个人技能

先决定技能应该影响哪些项目，再决定位置。

| 类型 | 当前官方路径 | 适用情况 |
|---|---|---|
| 项目技能 | `<project>/.claude/skills/my-skill/SKILL.md` | 流程依赖本项目配置，希望与源码一起共享 |
| 个人技能 | `~/.claude/skills/my-skill/SKILL.md` | 自己在本机多个项目重复使用 |
| 插件技能 | `<plugin>/skills/my-skill/SKILL.md` | 已有插件，需要随插件分发；通常带插件命名空间 |

`~` 表示当前用户的主目录。在 Windows 上，个人技能路径通常类似：

```text
C:\Users\<用户名>\.claude\skills\my-skill\SKILL.md
```

本篇使用**项目级技能**示例。操作目录应是黑马记账应用的项目根目录，即含 `package.json`、`src/`、`src-tauri/` 的目录，不能直接把学习笔记目录当成应用目录。

在 Windows PowerShell 中创建示例目录：

```powershell
New-Item -ItemType Directory -Path '.claude/skills/my-skill' -Force
```

然后用编辑器创建其中的 `SKILL.md`。上面只创建目录，不会自动生成技能正文；若已有同名文件，应先读取再调整，避免覆盖现有约定。

个人技能并不会因为项目源码被推送到 Gitee，就自动出现在其他电脑。需要与项目共享的流程应放在项目目录中，再按 06 的版本管理流程保存。


## 7. 补充：SKILL.md 的组成与常用字段

技能文件由开头的 YAML frontmatter 和后面的 Markdown 正文组成。frontmatter 是用两行 `---` 包起来的元信息；它的第一行应位于文件开头。

一个简化结构如下，正文中的说明需要写成真实流程：

````markdown
---
name: my-skill
description: 说明技能做什么，以及何时使用。
---

# 技能标题

## 前提
说明项目环境、输入和必要工具。

## 步骤
说明检查、处理和复验的顺序。

## 输出
说明应报告哪些结果和限制。
````

本篇会使用这些字段：

| 字段 | 含义 | 注意点 |
|---|---|---|
| `name` | 技能调用名称 | 示例中保持与目录名一致，避免混淆 |
| `description` | 用途与使用时机 | 应包含用户会实际表达的任务条件 |
| `argument-hint` | 输入参数时显示的提示 | 只是提示，不会自动验证参数 |
| `disable-model-invocation` | 是否禁止模型自行调用 | `true` 时由用户通过斜杠入口明确调用 |
| `user-invocable` | 用户能否直接调用 | `false` 时不作为用户直接运行的入口 |

当前 Claude Code 文档说明这些字段并非都必填，其中 `description` 特别建议填写。为了便于学习与迁移，本篇显式写出名称和描述，不依赖省略后的默认行为。

字段名必须准确，例如 `disable-model-invocation` 中的连字符不能随意改为下划线。YAML 出错时可能导致元信息丢失，即使技能正文仍可被读取，也不能据此认为触发设置已经正确生效。


## 8. 补充：补齐前提后的 lint 技能示例

保存目标为**应用项目内的** `.claude/skills/my-skill/SKILL.md`。下面是基于讲义流程整理的补充示例，不是视频里的原始文件。

示例把技能设为仅手动调用，便于明确启动检查和修复。它不会为缺失的 lint 脚本自动安装工具或改写项目。

````markdown
---
name: my-skill
description: 检查当前项目的 lint 问题，分类报告并修复已授权范围内的机械格式问题。
argument-hint: "[检查范围或修复要求]"
disable-model-invocation: true
---

# Lint 检查与修复

## 输入与前提
- 用户本次要求：$ARGUMENTS
- 未指定范围时，先说明将按项目 lint 脚本的默认范围检查。
- 在含 package.json 的应用项目根目录工作。
- 先读取 CLAUDE.md、package.json 和相关 lint 配置。
- 如果没有 scripts.lint，停止并报告缺少入口；不擅自安装依赖或添加脚本。

## 执行步骤
1. 查看 git status 和相关 diff，识别已有改动，不覆盖无关修改。
2. 运行 npm run lint，记录实际命令、退出状态和问题信息。
3. 将发现分为机械格式问题、需要判断的代码问题、环境或配置问题。
4. 用户只要求检查时仅报告；已要求修复时处理已授权范围内的机械问题。
5. 对可能改变行为的问题，说明位置、原因和影响，由用户确认具体方案。
6. 使用配置实际支持的修复方式，不假定所有 lint 工具都接受相同参数。
7. 修改后再次运行相同检查，并查看 diff，确认改动范围与任务一致。
8. 没有实际运行、命令失败或仍有问题时，如实报告，不宣称检查通过。

## 输出
- 实际检查范围和执行的命令。
- 各类问题及涉及的文件位置。
- 已修复的问题和仍需处理的问题。
- 复查结果，以及没有覆盖的内容。
````

这里保留了讲义“运行、分类、修复、再次运行”的顺序，并增加了三个操作前提：脚本必须真实存在、修复范围要明确、结果必须有实际执行证据。

技能中的 `$ARGUMENTS` 是 Claude Code 的参数占位符，后文说明它的作用；它不是要求用户在 PowerShell 中设置同名变量。


## 9. 调用技能，并区分终端命令与会话命令

启动 Claude Code 前，应先进入应用项目目录。终端中的示例命令为：

```bash
claude --version
claude
```

前者查看本机版本，后者启动会话。这些命令需要前篇已经完成 Claude Code 安装。

在 **Claude Code 会话**中查看技能并调用：

```text
/skills
/my-skill 只检查并报告，不修复文件
```

`/skills` 用于查看可用技能及其来源；斜杠命令不是直接输入 PowerShell 的命令。不同版本或客户端的入口可能有差异，可先用 `/help` 查看。

调用后应确认 Claude 读取了预期项目的配置，使用了技能中的流程，并报告实际执行情况。菜单中看到一个名称，只能证明入口被发现，不能证明命令运行或修复已经成功。

当前官方文档说明，普通模式下会监听个人和项目技能目录的变化。如果会话启动时顶层 `.claude/skills/` 尚不存在，新建后可使用当前版本提供的：

```text
/reload-skills
```

若版本没有该入口，保存后重新启动会话再检查。不要把每次修改都必须重启写成所有版本的固定要求，也不要认为改了文件就一定会自动替换对话中已经加载的旧内容；需要时重新调用技能确认新说明。


## 10. 手动调用与自动调用怎样选择

讲义写的是：输入 `/my-skill`，或在对话中提到“运行 lint 检查”时触发。当前官方机制中，显式调用与模型按描述选择技能是两种入口。

| 配置 | 用户直接调用 | 模型按任务自行调用 |
|---|---|---|
| 默认设置 | 可以 | 可以根据描述选择 |
| `disable-model-invocation: true` | 可以 | 禁止自行调用 |
| `user-invocable: false` | 不可以 | 可以根据描述选择 |

本篇完整示例设置了 `disable-model-invocation: true`，因此应使用 `/my-skill` 启动；不能一边保留该字段，一边要求“只提到 lint 就必须自动触发”。

如果要演示自动选择，可以先把技能调整成只检查、只报告的流程，再移除手动限制，让 `description` 写清使用时机。自动选择依赖模型判断，描述命中关键词并不保证每次都触发。

对本例而言，描述应区分“用户要求实际运行检查”和“用户只是询问 lint 的含义”。例如：

```yaml
description: 在用户要求检查当前项目的 lint 问题时使用；只运行检查并报告，不用于解释 lint 概念。
```

任务范围越明确，越容易减少无关触发。不要仅因为技能具备自动触发能力，就把提交、推送或部署等额外动作夹进 lint 流程。


## 11. 补充：参数用于传递本次目标

调用时，技能名后面的内容可以通过 `$ARGUMENTS` 进入正文。例如：

```text
/my-skill 只检查并报告当前改动，不修改文件
```

在完整示例中，Claude 会在“用户本次要求”处收到这段参数。参数让同一个稳定流程接收不同任务要求，避免为每个临时范围都复制一份技能。

`argument-hint` 只是输入提示，不会自动把“当前改动”转换成 lint 工具的文件参数。若项目脚本默认扫描全项目，技能应解释实际检查范围，不能报告成只检查了某个文件。

**参数不是任意 shell 命令。**不要直接把 `$ARGUMENTS` 拼接到终端命令后执行；先理解和校验范围，再按脚本真实支持的参数调用。

本篇只需要完整参数串。按位置拆分参数、动态注入命令输出等能力不是讲义本例的必要条件，先把基本调用和结果验证完成。


## 12. lint 检查、自动修复和功能验证的边界

讲义举“prettier 格式”说明机械修复，但实际应该运行什么命令，取决于项目用的是 ESLint、Prettier 还是其他工具，以及 `package.json` 如何组织脚本。

先读取脚本和配置，再决定修复方式。例如，**只有脚本底层工具支持 `--fix` 时**，下面的参数转发才有对应含义：

```bash
npm run lint -- --fix
```

`--` 后的参数传给脚本，不代表所有 lint 脚本都会识别它。Prettier 常见的写回参数和 ESLint 的修复参数不同，不能看到“格式化”三个字就统一套用同一条命令。

复查至少覆盖两件事：同一检查还有没有报错；diff 是否只包含本次需要的修改。如果自动修复改动了无关文件，应先理解原因和影响，不直接把全部变化一起提交。

| 检查方式 | 主要能说明什么 | 不能单独证明什么 |
|---|---|---|
| lint | 代码符合已启用的规则 | 业务行为全部正确 |
| TypeScript 与构建 | 类型检查及构建流程通过 | 桌面交互和数据持久化正常 |
| 实际运行验证 | 指定操作下出现了预期行为 | 未执行场景都没有问题 |

技能只是把这些步骤组织起来，不会消除每一种验证方式的覆盖边界。下一篇再展开课程中的测试技能和测试角色。


## 13. 补充：在黑马记账项目中怎样落地

前篇已经确认黑马记账 `package.json` 的脚本包括 `dev`、`build`、`preview`、`tauri`，没有确认 `lint` 或 `test` 存在。

因此按本篇示例直接调用 `/my-skill`，可能得到“缺少 `scripts.lint`，未执行检查”的结果。只要它准确检查了前提并报告限制，就是正确的停止行为；不能为了让示例看起来通过而伪造脚本或输出。

| 讲义中的步骤 | 迁移到现有项目时的处理 |
|---|---|
| `npm run lint` | 先读取脚本；不存在时报告缺失 |
| 自动修复格式 | 先确认已有 lint/格式工具和修复入口 |
| `npm test` | 不是本篇已确认存在的命令，测试建设留给后续课程 |
| 需要检查已有构建 | 可使用前篇已经确认的 `npm run build`，但明确它是构建检查 |

如果只是希望把**已经存在的构建验证流程**保存成技能，可以另取含义明确的名字。下面是简短补充示例，不能把它称为原课 lint 技能：

````markdown
---
name: ledger-build-check
description: 检查黑马记账项目的前端类型与构建结果。
disable-model-invocation: true
---

# 记账项目构建检查

1. 在应用项目根目录读取 CLAUDE.md 和 package.json。
2. 确认 build 脚本存在，并说明它实际运行的命令。
3. 运行 npm run build，报告退出状态及错误位置。
4. 不自动改代码、安装依赖、打包或发布。
5. 说明这是类型与前端构建检查，未替代桌面运行和业务验证。
````

按第 6 节结构保存到 `.claude/skills/ledger-build-check/SKILL.md` 后，调用名称才是 `/ledger-build-check`。本次只把文件内容写进笔记，没有在应用或用户目录中安装这些技能。


## 14. 补充：附带规则文件与脚本，保持正文清楚

最小技能只需要 `SKILL.md`。如果规则和参考内容变长，可以放在同一技能目录中，正文说明何时读取：

```text
my-skill/
├── SKILL.md
├── references/
│   └── lint-rules.md
└── scripts/
    └── helper.py
```

这是可选结构，不要求为了凑目录而增加文件。`references/` 保存供阅读的规则，`scripts/` 保存确实需要执行的程序；两者作用不同。

在技能正文中使用明确的相对引用，例如：

````markdown
## 参考规则
运行检查前，读取 [lint 规则](references/lint-rules.md)，确认本项目的检查范围。
````

引用路径相对技能目录组织，不要写成自己电脑上独有的绝对路径。脚本需要说明解释器、依赖、工作目录和输出；把脚本放进目录并不会自动执行它。

当前 Claude Code 还提供 `${CLAUDE_SKILL_DIR}` 这样的路径替换，用于定位技能自带资源。需要时按官方文档使用即可，本篇的基础 lint 流程不依赖自定义脚本或动态命令注入。


## 15. 补充：工具权限与技能说明不是一回事

`SKILL.md` 负责表达流程。工具是否可用、某个操作是否获准，还取决于 Claude Code 的权限设置和当前环境。

当前官方文档中的 `allowed-tools` 用于给列出的工具提供调用时的免询问许可，**不是“除此之外所有工具都禁止”的沙箱**。未列出的工具仍按其他权限规则处理，不能靠一个字段就宣称技能只具备只读能力。

本篇示例没有增加 `allowed-tools`，也不为了避免运行提示去放宽全局权限。需要限制行为时，应结合实际工具权限规则，并在正文中写清任务边界。

同样，`disable-model-invocation: true` 管的是谁能启动技能，不负责判断里面每一步是否安全或正确。用户主动调用后，依然要遵守任务范围和项目约定。

技能说明也不能保证一个规则在每次工具调用前都被强制执行。需要按事件拦截或强制检查的机制属于后续 Hook 章节，本篇只建立这一边界。


## 16. 验证技能：检查发现、调用和执行三个层次

创建技能后，先用小范围、可观察的任务确认它按说明工作。以下是本篇示例的学习检查项，不是本次已经执行的结果：

| 场景 | 应观察的结果 |
|---|---|
| 打开 `/skills` | 能找到预期名称和正确的项目来源 |
| `/my-skill 只检查不修复` | 检查前提并报告结果，文件没有因自动修复改变 |
| 项目没有 lint 脚本 | 明确报告缺少入口，未假装执行成功 |
| 项目有 lint 问题 | 报告实际位置和分类，不凭示例虚构错误 |
| 明确要求修复且范围已授权 | 使用实际支持的修复入口，随后复查并检查 diff |
| 只是询问 lint 概念 | 本篇手动技能不应自行启动修改流程 |
| 修改技能后重新调用 | 实际流程与当前文件内容一致，而不是仍沿用旧说明 |

首次验证时，可在已有项目分支上使用可恢复的小改动。先按 05、06 的方法保存基线，不使用真实账目或其他不可随意覆盖的数据制造测试案例。

判断技能是否有用，要看它有没有稳定检查前提、减少重复说明、如实报告限制，并覆盖原先容易遗漏的复查步骤。不能只看回复是否沿用了技能标题。


## 17. 常见问题：入口、触发和执行结果分开排查

| 现象 | 优先检查 |
|---|---|
| 找不到 `/my-skill` | 目录是否是 `.claude/skills/my-skill/`，文件是否叫 `SKILL.md`，会话是否在预期项目 |
| 新建后菜单没更新 | 保存文件，查看当前版本是否支持 `/reload-skills`；必要时重新启动 |
| 能手动调用但不自动触发 | 是否设置了 `disable-model-invocation: true`，描述是否准确 |
| 自动触发太频繁 | 缩小描述中的任务条件，或改为仅手动调用 |
| 调用的流程不是刚写的版本 | 查看 `/skills` 来源，检查个人技能与项目技能是否同名，以及是否仍使用旧上下文 |
| 元信息没有按预期工作 | `---` 是否在第一行，YAML 是否有效，字段名是否准确 |
| `Missing script: lint` | 当前 `package.json` 是否存在对应脚本，不把配置缺失误报成代码通过 |
| 输出说已完成但没有检查证据 | 核对真实命令、退出状态、改动 diff 和复查记录 |

当前官方文档说明，同名的企业、个人和项目技能存在来源优先级，个人同名技能可能覆盖项目入口。遇到名称冲突时，先确认实际来源，而不是反复改写已经正确的项目文件。

官方也保留 `.claude/commands/<name>.md` 的旧命令形式。它不是本篇采用的新目录结构；新建时优先保持一个明确的技能入口，避免同名文件让排查变复杂。

排查应先判断“有没有发现文件”，再判断“有没有加载技能”，最后判断“工具有没有实际成功执行”。这三种失败需要不同处理。


## 18. 保存与复用：让技能跟随项目演进

项目技能可以与源码一起版本管理。先检查技能正文及引用文件，再按 06 的流程提交和同步；不要把个人凭据、运行日志或当前电脑独有的路径写进去。

查看新增技能时可以先执行：

```bash
git status --short
```

未跟踪的新文件不会自动出现在普通 `git diff` 中，需要打开文件检查。选择暂存后，再检查将要提交的具体内容：

```bash
git add .claude/skills/my-skill/SKILL.md
git diff --cached -- .claude/skills/my-skill/SKILL.md
```

上面是课程实践命令，本次未执行暂存或提交。

当项目的 lint 工具、脚本或目录结构变化时，要更新技能里的前提和步骤，再用第 16 节的方法复验。稳定的约定留在 `CLAUDE.md`，专项流程留在技能文件，不把过时命令一直复制到新项目。

可复用的核心是正确的任务条件、操作顺序和验证方式。一个流程在自己的电脑上运行过，不代表复制到任何项目都具备同样的脚本和依赖。


## 19. 来源与本次实践状态

课程来源：

- [VibeCoding实战网盘](https://pan.baidu.com/s/1zQT5QFTIAOSm-e5cnSYy5A?pwd=1234)：课程目录中第 16 节为自定义 Skills，后续第 17—19 节为 Agent 与测试角色，第 20 节为 Hook。
- [Part2 工程化控制和Agent开发](https://www.yuque.com/xxcls/vibecoding/ia0yu2m1v3sl537p)：读取 4.1、4.2 正文及原始代码块，保留指令模板、目录和 lint 技能示例；未声称完整转录视频。

用于就地核对与补充的官方资料：

- [Extend Claude with skills](https://code.claude.com/docs/en/skills)：目录位置、`SKILL.md`、frontmatter、调用控制、参数、附带文件、加载与排查。
- [Commands](https://code.claude.com/docs/en/commands)：`/skills`、`/reload-skills`、`/help`。
- [Settings](https://code.claude.com/docs/en/settings) 与 [Settings reference](https://code.claude.com/docs/en/settings-reference)：设置文件作用范围及技能相关设置，不能与讲义的模板注册示例混为一谈。

本次只新增本篇 Notebook，没有创建或覆盖实际 `.claude/skills/`、`.codex/skills/` 和用户设置文件，没有安装插件或依赖，也没有执行技能、lint、构建、Git 提交或推送。正文中的命令和结果要求用于学习，不是执行记录。

讲义相关部分以文字、目录和代码为主，没有需要保留的原始教学插图，本篇不添加装饰图。
